# GwenLand glcuda Wave 122 - elementwise-stage T4 profile

Event-timed production profile with the former elementwise bucket split into named substage counters.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave122-elementwise-profile-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "7bb5148d5c7c8b4fdd096b47204d0ec40cbb0bdf"
PATCH_SHA256 = "a42910a4690247adf20deb7a22d154746278b4cfd485ec385c28992122b3d4f0"
PATCH_GZIP_B64 = """H4sIAHTOp2oC/9V9a3fbOJLod/8KxLObkVoUrQf19Lhn3Ik727fztJ2e3U3n0BQJ2mxTpMKHH5PknPsj9hfuL7lVBYAERVJ2evruOesPtkUCBaCqUG9AXuD7rN+/DDLmHFyGbu45B/zOWW9Cnh7cOjd8OJzbQWRvktjlaWqnmbMKwiC7N5OUrb61x17Eb5kfhJytY4+z4WAwtay9IPL4HRs88sc0V+Mx98Zj13K8yWo8cofugs+ms/HQmViu5S4Gc3+88vzpXr/fZwcevzmI8jDc6/V6v2PGf/sb6w+MAesNjeHEYn/7217v4OAJ+zt0Y9CPBVFf9mPw18vdLIgjVoBgV04SwUuTuom+byKx/tBgSR5FPDHYs/fPj6F/sHaSe+bGUcbvMoM5kcecMIxdh4A6YcaTyMk4y664AJXwzAki7lFTj/s8SbjXT3gaeLkTso2TXaUmexav19Bfm58fOpcpcxLO0nyzCQPuCXire4TNXBiVJ2zF/RiaqPXBopLsUDbIU4Cf3gaZe8WClPE7gOICF13xhMNi93p5yhkgGwAslzy6hFnaWeIE2XL5+UV4Qg8M9lMEU/4p2uTZ18OiC9CHGuE/z+LIDy4NJj6JbmVTnNly+YL+ineHODQgMM3Y29M3r96e2+9f/3S+ZE/TLGFHbP8Vd9I8QQzCpD0OCF0HUZBmgcvS+zTjayLjepPBEhPuA9/cm+wEFgdoZlfxLcviaw4kdxJEUQjkz/gloGrtZElwx9Z5mAWICUExmCWgDVgAKLTm6zi5N9g5j9I4AZoAlQSJ/eAO3odOHgEuL3m85lkCo+4fqpW8e398en5yfrYEgME/OKxjMihe/v349NX7t2f225NTG/7V2hRNTv797cmz85PntkTJ+ZufT15r0EaWRXjzI1gP0KITeCmg7EM+Hn3ssv73GpnY570eg5/6E/wh5NjUG36ZWWzfcLfTNcoWa+fOBiFgU0toNtTeAfY3PHEyoM+SDcyB/ire2NdLZm0/22DDxUR7mvANdzJ7wyPYLvcwgKkPYZrlxJdL2DAOEKzTFQ2+7vW+SjTAvrSdZN0RLwT7AkZ0LpRQNVTJJy7QNPBgmy7ZKo5D+TROHDeER9AQnhBWT3kKo//Fn1oG+yG++4t3j4LDg+2SJHGyXJ7gn++/VwgWszBTntkrDqwCsuLapj1v+35kq03fKcbv/vVQ9Ax5xmKg1JGCESASOgWtu0XLwMeGptgCkkjsyVELB7EvX6h5QXbTAekJ3M87Xez1QSz6o84iILHyJGKwtg4IF9g2TzrlS/zZF53YOkjhrXu1BDG1Pvr8lVUmhQ/Uf8u/fkX5w92Me0cfPn/9uG9UQcKqCqSwz2y/+LDPoGeY0kMlSuHZVvcaRhreFyjYfkeL0R52NWYE4RF3ut3Dgv3wz5vrjhiQg6IM7XXaLdlyDTPs6LwDe+sxrIMMsAYGcBKQ+keyZXSzXOKDTtdMr4NNZ9jV2IX0EzTFBtqMI9BLHX0J8bUdJ5190AaXwNy7lCl79eb5yUv2wfmy+rhfYU2yBcRQcgDtZcJveJLie+IGnj7pYHvkNA9UjY8YOAOR2dlf7Xf1jjABtDTsOHIf01s2FzAqUFAZHOnaxBQypjOvIAzQe7nJoaUuJJZL0JBXtktarKOrNH1PpJx7SzEPa6RLy137XHbwHeBf1eermhHMBLgrAFT+VXsSxo5nE2U7T+lPhQzAvDB7bAejwlvbvYJFPhUY0CUEbvSQR2KPt0iGb9rxSuOC3QPbJWNeEvjZksEWhwE+f61tZzX+1vPmqTxy61Up6UpKlobFcglqq+BMt4Zf90H8ClFgC0EsAJRS+GlFDJfNoel3ZUdtAYXA0Z75QZI2bk9hKSmQlzxCNQs2RxQLIbpfWUZ6lWdefBt1yq0AhNNEKJhCH4jvDJYleVW+Y0sbW4ChvmWa6M2IMaSSfQo8ZzDEgFEOYsjZFjPTRGRBroMD9maV8uSGzK1+HIX3oOLAemKbGChMk5GW+sJkP3O+YTEY37B/YRdBQ+h2wwtQ+gKBMhwFBywE7d3CBI7BYgNlQ4Z4qGQM4QZ2cQgqqIAWZBWP4Bp6gKphKA+QBWAOoXMfRJcEfzQY9FPySDTXAfBjFpuuIs40RAohpXQFsEsTVpFKjQjF3hnMm0xOsfurNDKLtzpf0RvAlg1zjzpfsi+sUFjbrSQHCiqMBmqu2qh55Nw4QeisUPrqkwN/KMrCqCYuPpCaGQ36EifAf59/3a9o6F/3l5+/Gr/uX8Up2IIFevDx0lzgG1jpwy+yjXwzxTdyH9IINMDXR4umcqStFwUaYENnTkODUtg6KQNTkX2HbjNYx+zg4c516+OwulOB2S4Fkxeg6FG6vVdbaFGlB3UV1IicNQcc/br/+euv+4A7NUEN0ehkFoRa3Wc8Bd3meOrJ2nHTFiyTzsSxTBym9WULUsoGNIPWt+WUzDy6TZwNMvKg29oeJ7yrZbcuyJRirkjcLb0J9mCLoopQm6KRpgtj9ZsefawbUh6215oUf0iSV/QPWEjQGGSPssE+l0AKu1lO4qtuSEQ3KmSx1R1sNM3idvaLbsiLn3Inyag76g7l827rls4mTgOEremKLvah+ZJetwMQz2DV8ihfk6KDzbPFzo+Smrt1kQIDvcQyv8nFOPzW7TWcSwUh9leJY32XSRSqLaRQpT7DVPXWjRKzUSb+bnlIEqyYasNLOeOGNwWd669gHU3jtMjenfL3IRG7s2NNvLZsc/mngcglcWGLeIGb9ckcfASNC4IJrkjtDfgHgsCSjg3EFROW0A4QYJVkjaRSO7HhEWBqpHdvoIDW1Nr9WPddagJRSEH0hL2GiHWauAfSwDoAw7sSm66/kwHn1cjyF6uF406H0/FisPJW8wF88Mfj0Wo2HK/43Fu4c39imtP53Fr4rjeczxzXmvvjyYh7rjvlswk8scaLyYBbw4GjAtoYd94xt2okuuE9xpynxpT1psYMI84MQ7Nk+AGRUQreOonHNg4YpB0QifgQVFXWNffYHsYchYPv+8Fy6do3ceDJgCk9Tu8jFxz/LF4H8PfzMf3zAwaq2BsUoWCSgnSSgFQA90VIEQV8jpMbjSYws95oJCbINvkKgCdg6oKRjUs54xgXFPqNr9d2NJySUcUpoCPCYkwYygfKQh8uRQSbcce9Ar0S9f0Ajewff3zNiog2yneymDFQwMCC5smf0xKUE6K+vu/7OSopJ8vAyEc9dPrqrPduTigz2ZsNbjOwO7Mg1E3044MfzL0+gmrytrVYXtNrDY/bK5scY4R8O1SPkcdDEYcGioJ3QctK4lvpKrCV415rsG6vAhFqvwW3HL0QmC5zLgFmCjoTrAHOYfuJ9gAltQmqpyZOZFuMjeEQ6LawjNHiIcL5NuAPloiwPl1bGNFZ8aVsBxP7CsxAsVKexuENbwpFSuUOxLA/ze3LMK/GRHl0EyRxtAYi2TxCB8CrvC/DHzGIxyTwoDsQD2j1F2z1vQqk4gel4fXB2NOnDTA0G61hAjLaxgKQqnWcICH+z0/nRCq+XnHPA4K+Pf93IqpEBENzp+KHAsuBe7i2Z5MepW4KUBnlAvq4xzAKloP7BwTGMAKAzeKYdS5evMTckP36jf3q1fHR8ILFG3AswYM0kG1KUMgNxVrRrVSsgVPFsQfM21gOe3Hy6lVXsglSHwiI43Uo68KePoM/eoCxwACmYERUUbelznjoL5cU96BAl+ABhWkCarDX4HN3a577AXsJ3Wh6kuHTHAw3cpCZwyInwb1QLEk49wpy4dmX0ECAmOwCx7pAJKL13odWffxH+fBqp2t0BwqFgXsvHe0/ffBit3MVAF2j7keptkssNS6yxIaGQ03V1fdFOzuTHnwQ/RX7U/LNEY1ehKDgUQf4Ei3WSusUjVWiC4afYhM4Y+38FicGqz4LwK7vbvVdrx1h6QKQ7wHKzGCTLu6xMp584wBi0s5+hWv3u2aQ2iDQhSGOkmg8nxtz1rMGMxRI8KR5uxHLSPN5r2J6IZ21NodCbquZ1lHOjsoWTWKidQnPT348ObVBB9mnJ2c/PX9//FIsJ8Xga3crlNI07sPysWlOW0bl75zeFpQdnFiNEVTIrimTSu5gaybH5+cwiTd/P6viZ5uDLLvUykf4oGy8kw4EHpjJegh8wi8/6WMQv1nDBRosljUUBks7u+kmi9H0GPN3ja0LA8eoslod6w+SRTcnROC53qa7NQ2NTNsT/OTMMJgfYOaq8qZKD2EkDGdDC3E1nI2nxnDeji0ycK44iNZEs9ZUfI9Ms5StnXtp1UG7IKkac47nkUGnKURl2TWbbhW11cDNUod3nqagl2qmgUi1hL5Z70lys6Ps3+XylIfOHRkDNa11JmoelNEKGiq75Rz0O9jVV6BUgn/APinUFsNYBHlUqVnAKIG9LXUSpeoT+SFCG4LKMFIyEU12Dpgpik5kUQkDDyllTgkuhcmDLhBVJQeUgjiQ1SRkoVA9ApmcaBqApcxiX8W2+5dhvHLCEpiuJNd5Rot4WE0+mJ4m2hisYu919R3YQqKtuHKagdHUobbbslyCLv2Zkp5SYKDpWvBcgdnRDIwPj/cvQSBK843cCFkU4yXBDXobLHbdfONE7j37lIOlZwrbejaeocs2ms0GxnCijGu0j+MNtzOcUorBK1l2YbArwL/tBevigZ/wT/bKScEs8McjYt/OL9z9C3z43mC/gL4DipKnlyp8kVeXgwO4XH6HzpmkDraRZIEJqKysqg+qS38bJKqq4LExrg8oBWrbl5glqkbPYBvyJHvSebJDrVXCi2T96YpSQXgQgPj9+/vLCVCaFJ90G8E8eeQ8tGzrN8OpxGcb5kMeh86P6FIEa4zNr/MU7Qo3xGItfue4WSjYUfMv0lI4l2DkdFL23//3v0CSUPZbZFYLkYVgzE12J0ryUECQbBDmo+Z5ypQVepjkWbAXb9+brYGYMFjVAjDimQy8LGY+X6zmjj/zF9Z4POXDkWXx1WiwmKwG3nwys6ajOXcnpjkZDFeD6WjsjSbOcDYf8MXctVxvsBj7w+nCW4wmq8ViNZ20Bl7kuLWAi3yOG3cyxG0Lv4dj3LS0oyikBnvq85n4D4gm/hG5e4yPaJ7zVlq/wNuPVNd1+voFJfjJZcHMI0r8YBXKYQChh8ptOWJZsOZ9bM09TdmJ+gDpKuRTS3kKOMgPDzhIqKP+nBa1gVUFjPWBTWpJzoeypMmNDE20+E8qay9UFEwci9+wnnArvFNqLdJCF7gNLjChuorR4UvWKWVR3ZjfBSllATBVK5VTCU0xo9BIwI1V9VWZGpiQAcndB9TW7jKLqosmdAfORDxHLPKkT3FWUtgYtgF8YRwKnXgwCfLLK/bholoWAir64qPQHIsFMuBwULG29NY15SjDhVjFQjUspVIrrRRKRouQkgo8aRxBlG82XUqDpYQms+AFcUualoZOWej6/8dCqBY9NQUj0AVQafYjwtSWzSBfbuexFRYbEtc2+p6dL1/UeMulIAmWVCCBwPBFVg2ywAkRe/uqsKSSKlL0enDlKgal9VWB75qaOKEiHXaBxt0Fy2QlB1XxUAgFCSYjSWQIghCgFjBNkGdgesPmqRreP7w5E01Mdqwy8szxgR+IfRXrllUuFx9FNXCCodS1pjFEUhAVlRwRNv0GpCV31gZbwe71gduyPo9gsqh2VDU2QHMydhWHnrCpwNfCMHMhnOXWkNsCRVzbPpFFGCgOUxAUANphm2DDQ9oJtzA55oYx7FTQGFpk9Z5dgNkjZnpRg0ZR3E2cZDKKi+1JY1KmGV8jO7jxekMVvzICVwMDywZ5gatX4mGTZ2bVaaTCDJm/Jq8bk/KmTGyunU3nS/qFpUWKu2um+boSjpBjvU2U8wXDworRQ0D1jUFKDEye5es16XTvN8dFmSnT/0HhV2jAQHD7eYi+WRIAFclrueLutaBpgDXTEWKgqK9hl85GUDjIatCgzU0Q5ynV7LhxnsDAHrlBmLZFpCpUFnU2/SDyw1K7aMCw0EOQFTlR7ABNYel47u3A86ae5JM4xrKQddrwuogmF7LikcTSoxf48/YKTP+3YqXss4RilLP7qsW7ukUmxqLAxmhmGcPB47aIjO5HJLc6T7HcrXCXmyVsBVsk9o7YM6rSp5wAFcH1m6VwYZgKudF5it1rJUit7R8IuOIPQdwOl6GWFEWX5i6Bq0e9tiOlAczmaREU1V6i6o+4OA2Rb1hIBxnC4FodSABB+yEN1t5HerVkWKNCGxA0pHcL7FwBRYoYy8rAeQ/J1Q7SviglQ8aFfXgTpGguthvcJItrJrd6Ko3u2WQ4na9m/mqysKz5YDW1RvPJyp3O56vJ2Lf4dLXwnNVibpr+aOD4Lrec0Wjgrqz5jI89Z+At+Go2dh1rMPcHHH7mrUZ3MXLN7C7eIOvORyTd8Y9m92zyV1R6rEeAq4QB6w/2sY06BkiUb2zwj2WmgZh5lYPx8BQFgqk1rdBXelmY7F7lIZ65AcL9VqL8xcmrX4Q93QmdtCgVoexcD70xmGJXC/Nh6AVJqNr9IEqhmJeLsyAgkOI8LIQb++X0+FVhyTry6Icmmf5JeEJOV8DpZ5M24E4A43bW4iyMJ5NXv1EBPdp6slhvqPJEGtevHfcKuLqP5i3ZB8jjYrKlKYglMWjLw6AASKjXdz//Uud74OTEifRgFFVeYswG34GVsAbAINHhgclO1kGGM1SZ8Ao8wAzIC0NGzmJhsF6BMhI+TtveEaGy2uYpHsvd40+ssT8djheuMx5Y4+F85vnz1Wg4mE1Ws9l0MHG8+XzsWKY5Ww38xcJy4dGQT90RlgvAXrJmFjivI9+zpquV70+GrbunHLq2fcpXFNOeY5yph3+2c7kveHROSMNtRAYZyKBUyieVJJe2TArWnpBRwDwpuo50wCuObmQANnTQTAsiAeniDQZ8L2S0gMNuc68pSRD7xAqXMQz0KQ+wMmoVC7NAt7HoDBXYTHsyxkhaCTQfsA9ar5GUdkym+Qu4IlsfxbdgKOFKxcmms/PjFyf26+NXJ2dL9gEPeR2y+UcscRObc//T9c2+sW2ADVifTFTYy+C4BqJ6GEuRYIx/cHj67uDng18oVZpKOJQKxz4FNEw5ApxV4ICS/nTdx5cGO43fnuhdrm/s2wQsAeoGXUbQ5edfYHeA4cToTdla4FsfYAytEQtlNBxTxbIHKjKq2oyy2yDFEaCHJZd2oFZzAGolPyg8foyRav2xxqbED/SfyBHxhSYUtS4YE7TzjewFXaayi6i5wNc9kC61voQOYIjKcLPaAoXI1rv3dpN7OBL07rXQ+/fSvddCd/Yw/Xst9Ne6NvBBr4UP2IP80Ksg2KYF6d0lXxQLrWEbzcS4r+O8hUMewSm9Fk7Ruj7IMb0WjvlWziEw4drGUHsDWwCYOYARiSvJGI02gbaoInaO7QtUA6QFQCq2WQ/TVq8FxHdzrTvuxkYCDZE5z4KX79UZ0vvtrsXI4LtXNtEQ+XANWExkFKIS5NtjH0W4suMmVCurtpINKrk8AzpoaXT8umwzPCTFMxtjwGo2k055U68X78te08M9WZuFwT4exnjmIWaokfGQhaY7hC5JUV9c3MZUxnFcJCfhETHjhYAG73CV0BSggr94KY9SrMmAVi5qUVUlR8ji2GxZ55tyxrNDEjq1Ni+107Xzljanr9+VjRYtjc5+eqkhaDhog3X8/LnWDNDfo8jPDdWtOAFYLVzaRKt7CpdS1hW8ejSCMBhaqgdpZKrzyNhaQBNWAGJPoN9AhIocIYaNw2BFNdTAj+jaX2giWNR6Xpjagesff3p5Yr/9t+OzE3Hq5+Xxf5ycaksYHaro6TmFTDx+E4AxR6X2YOnd8uDyKhMJtoCOR7tJnKZ4CJ6MwwwPoWNsBlPBGLGmhPVwaIwt1huCI0En9HX6YvqLgFIhf9q5xVO8m/zv9AysRXEgl6Whs6JKN/pMrnA+tZT15JKxLSzLlHOYx7II7EpbmUxZFRuamYt/FZlVYb6jrfQPnsQCGi3WKC0isJyhwW3gZVd9sJigrTqVodlCWqOCmyUO+iJphImbAD12wfJ+nCdILfvk5fkF22BsQbwgFmFhHG8wbgcTEcHdIBEV9IaAh1NGqyzDLQv++YYi8mCmb4ro4tpkp4hTho6SIQ5YIhsJC4+TGhXAZExJ4IdlieP7gbukRkonwaxJC+IONaRxGacFZwDDAvcJaECNCLEsbhroV4mtcTxR3CYAHcHqzUiitPE3IAhnJ+D9AQiSuNG34j+BIIUbOT1xFcNjEGSn4FhmHeFR4aaYWrJURCSoi4+f7K0HIsSvPyGtruo4+9peEsWaH+DDIbM+Kg8fXXrw4DH48wUnlxKsLzTRFMvDZWwJFZ3J3ikbplH3ox2DL3RbRgZW1clOga8jGrITAXxaUVcOEvjSKNEOTlftqqauRdNOg3VgsFLnd0X9nBqiYhAis2h1MUhhZUPg/4oPjCIzhuXfGBNjd8oZqwCkAwcKCQbbhLlgb7JzhExMtaCzwEtPrm4Eq8MVwvow4FGusjxNVFso29G3YsSaZOd0LoklvwPPfWuByJakjzAxeOVQEht9/gdnK9NNW1Mt0c8ebP8gAfdaahIFBTGolKIrfRtjGCSlPKAgFQe+BroaAv8dIoekQtcoFv8YgnQBqwULlkjeevoNBLMeJhhhAAxTIYMq+WADP0QYv+RgBshjCWo9YERkjyWbvoCCMv2t03LNBB3/EwTtyQLSwlPa2oigx3G1aGxKKUMZ3uIgglAXdGxHrlSusk44/TBeacs37Bu1gg9amUlFClUuPKkvUKt3K1ba1AUZUjz+KC+Z+NMH17/s0GmSj49THTIcr1RAr6o0ettKo1dTGr2a0ujpSoMOCuxSWiDcALwhRjEkbENA3D7yL48Gak8pFaNl9V/n6xUGEP1aqaLMTq4xpllQ34/DML5FaaXOopBJOhGxueF8IWvBVL0ill+hM0eGnpOAkdupnnyQhZPSZtYt0T99wGq2G955zlf5pcGegScF0vK5OEoC5NIjgG+F6Vmkk/QzKWHYF7lPPDhE9gcYJ9A4EDtYaQWEhtr4gyyA8KfW9xhdM8psfv31EK/BKcZ6rpv3SDh9SMD8GqO5qMLEkYlExJW18annUtoMlaErb6qjngcY7KRTRMKSxhhzRxak3jH3Ko+u0642CtVZNo5SeVMd5ZV00/sOeCsgGinJXCzPZD/SXRd9jB9vOPyKhGcVFAli7diSyn8Rg7344SBVmVQMDqvjEUot+vwW7+YinGKUHSzV6xIUR9MxQAcRHIwQwzPqjgFSTMBy2dWa031XwLjgLGrEdtxmLOgvqkh4K27ukBcFkdcoc7WwSsy9a1VM6sonUe2okSvJMUnAI0mtjMjn4mwpAUB3o1G6OgW9qXL4hpbSkHXLFtVfDiYyE/qo3eAlmKMX+kyCBlEr+VFu0ntlXyXiPjQZN4enGtrvuCsy4uLcNFFSeIzoRKDRVs7XBJ+DsiWiui917oU3qCGrqEKoHIgSJddxyPvKtaRUfuhs0KCkzDehDO3DMt3Pi3iByd5HmKnUKr2KVAEmcwnmhSEIye/cMPeksqf8uou/Yr/gRLqGrsAklhmBHNOGS2X1CAJAiwgzNOQ04Q4ANYhFbr2SPco0e1F05VOxWymdn8eqgFnYTn8WuTohW3AYEMxzPbibovpyr0Sh1WiACcfFZPJAwpFq9yIsBBaFU6gJ0nzVV/GpDqUG19yJAAtYEVEU3BQXgsRRd1kBqKKcFFw2RNQU/zeKTIoerCGftEN6WwTwK7AowA8aqUu1naCJ8IrAUHlFk2F/OvtXmjQRTKRzHDyBTwqTpeCA6qY/eS7oG2X2ZW4w8Z8Xqf94SH5Lp2qE7e6g/g3XW12LwyzIrMvl81zUmi2X/3ly+mbrdML/YDtiD2uBkqO3mE8qtRTNDIK7+pCp43nnJy9PXp2cn/4HntBz0uu0cEXpTh6zmsG+BbLaSk4eEdMwqhtpOftSQN9x9mULZuXj06fidJcQonZx3Uqni8Niq636DaoJQWFsNxkAmG7BAlH8v7Wj8rM/DKQ+aW0pTjk8piXqoWrD3mNnLbNEYtrwobXr1rx3Nd2a+K6m1ZlTS1ZhpzOl+ihAhJ7jbSyu7ywExH0R5eGfcpBvWC1SSdALLp5O8WTdcDAYGUPrIS4uC+EMtuJ4qyeNQGEq1N+oWUx2UdWfy2VTjRxdoJSgRStDm7J8F3QeFwIuItNaXKJEEhv9plRqyZD7WVVg5jxF9WVWcSqsN9w1phfc2C4Pws7b05Mff3r50v7h+PzZv9WOr+FaKOxS2ROfq/V25f6wsQM5Y0P85ZqRLUzHbq3o7BxoVKpZLH1IRZxFxGcCOqsvbQkKMKIZQB9roGiIA1qcvKhKFNxKnSpgh2D9y0Q40k3q6xosCj4UKXVQXCJHcCiMHaqZxsNFOBaTGTOhtFDJ16AhG2D+DKs0VHWAVk1yW7oTeBdqAnxp1mAce4IDEC8i2/byVZ+AEpb0woMy+roFpk6hHkMCddrSDhXi4QfBOT027Ha/+XipKNSei61lDfRTq01bq+ksZVG4V9Y11eQclc8csQHZyA3SRL4fyfds+z16GmjLMTw31JHnivQ/3e+hO7ysXmcnQ58IhU6nJKi1EctFcXTt7jYzAWcx8WTh1aB660l5wBATOHaShzx9IrfAFpY6/yKsqiW/2yQG+xeSl+rDKvbulytkLpjQ9+xzA4YfnDb5BqM5lUkOx2NLOwC6i3LqJ8IwXl479NlYNqhTuV+7ggZX/6QjzSORmDG20drY8PT1O6NtirD868oBuOIMZHfXqq7NZJ3aKkqHfTH51WnvoHagPCM6tsYCn/P5/z58Yvb1j0eoCpHqWP0GhM6oGHJoDWf/Cxn0+PnzVoRWLni52PiogX+7UMXyWZy7V3Rgllxrdf63FZSMsjWfD8akKubEYnUyp6yBaAGHlYqgDPFeUnUwFiM1AZh2GJSlALCIo1zc4ZwlsRbgIeCZHQs0wWjUSK5a1f/L+BLz1+Q6ls6BvEWyCKLIglMsNJWBZ/pMuksvx5SsiqdFQaNF7AgMom2bRikGgnDE/E8dwP6dwToR64MSLELN/WbpgF5mp/6yZNzGVzhYyyuqytZKck1vkyVtbekks30XtbzHXIXi+8YGuAK+SRveVqrVtUP8N/at1GZgDNMZJPM2Vf+JaRvavMBGQYLW7oSr7JBwbYjCkcaN9FgC9dpFuCBSb7eEaX0tiNX6uoVg7e1LorW20QnX2qggXq9NpDW++GOJ+LUJ83U3onEmmi/5gej/EbNeshJFpE30KZm3bVQu3UcNjt4TU0IKp3R1nf4yiLR3TUvc2/UR47s86QtHQZbmyNoEQ490n8tiodLHo/BnDRheZ5myfQp9QfPpVC+S2Rc5FfGWanWo3EemSJmM8VrjoTHCL+Ow5iNjNHm0oizUxXL5y8uipuwZ+DvgzPqdp5RdQ3/e0JuKYxgd5KSiQe3YTp1Kx8+ISqILHkPAFx3czsIXaQVQ8EsNgpZu0+H0H4ADGpwAtWXu+u0GBWbmdkre1gaf7AebiBzd7jZtVliTRNdQ0msSsB+2sqfsgaywUUmRfmRHzZtzdy1Nb7fFNrUeISV3NqoiurXZFrJ77SZsG8KbpfBhy0W4dfbbqqDZQawtAOD5EICm4oXHQ0F7X8jNsnDgG+YAxm11EmgU7tjLxUVQeLTmqFketeWCr80dt9p0DSHcxcii+rLbMoGis4pniiKPQkvoaqHJMD7B7GmR//KSeCPkOyAHHmHmGW3jP4s4JRU/CKHdDK35Wh9RKkKHZOW17DLZTklgioK1QpNlb6rM11PnRYo6Prwj0c1yOoUcpGnOq5Uouzi2f0SBpS0UflcQ7bE88zCch/TAu59/qbLeI0AU3/VBlSVfQMOpANBa1L6+crIvbXpSHz/NGoyWNZgqMH6lphYfGiyKtDLa7qHU19OB0NeT4RT/eYy+rhslVbJhGhADmcJfI+sE+FBkHSlOaorzmfeRWzhat5g0FW2NGjT0Qyliq0waTLtjLxF1veV480r0Kec5pp1VMpPKo7ECowZPHKcrUqRrPLVEqbiUTnnSFU23To0dH4xk1ZgXV/e0k+I3hRlsRbdgP5URwKbm24OsUxrFlPlje512BJyWvpqlI7jjSMDRn+kXoJsDrOlap00u5lZZF5VNkS97xKLme9IVX+t5YpgBZXT+oCjmsOEA/VlxN4aqo9AS65GIzGMGRVUPF4cqX7x9XwOmIuSinBEL+VG46kF+5CqVHS+y50GKpSM1aKIoWdjKsl4kVWdA8LIP/JoKmVsusu9gU6MK2foSgipKKxwxaEDLbr7r/U6e6/2B/NbAa/V4l5ix6YbcSTpNTKqi8dtB/L1+ta3GuU1SrO4z1r1sdc94mcGlpdYKV4RUnY+NBQrV8eKxiQhRwlP6KQ3RQlWfs2l4VyT8GoxBnXuMnekOHSOU9G64ld/GW5DscL2F8/IVGpOHdEIa7z1L8aokdUMb3l12kNnXN/DLcfWSjTZgYGe1wrrMD7Cq4YDinLuBiVmJ6UXAgzgH+uO4UhHiZYp0qeKiLQGx6243jaArQcOMvgvG91OYj6HOgadovDtYkttiXxotLoumw/AiSzlGeuVssBx7NJnaG+deD8fs8nyM3TCYXm+La6QC0+VyleN5sOXyOb85w6+Qq7eg2wOWy+L4jkLtbCSisfPxQB6FruESa4fEl0/JEHEKvjyXGdVAVC/jdzqi0C2/cFOIVixDF4c+ClDulSMOnAVZGdC9+JReyHP5eEkJim79vraz+FhOIZUHoMuLFukQNF0fmqjCnMrNhMPxMVopfBXH11RJlMoT7qm8E47sYXH6I5UHuOR3jJbQVJ0R9lf6QtwH58DSM3n/BH7f5U8+IMSHN1flCSKMfWiwNhsQmqmsFENUMx+UWCrKXYqhijnTATH0u1SYCGP8/fLyoUCcCcw3+JUZ5Y1kGgZGo9+Ngd7vwsDWUSByU3RQsbj41MEa1vII0WMxUkIqvvVMQ00dI0y/2FFd4KLvQrEBnYTjdzLZn255JNx4GbCjp3QgV3xTYlpNZ8E83kGXkTnpD8zJD4YsiIyw+m1kWfJ6M5GoMMhYFlHBaunF3tZlSZPhqCCFTHLIQ3QoJ7AEt1rXT9VpR7vDKjAbg80XU/nLmuO3Z9JtirU7GW3+6UlHiqQP85llW9hjaM8mI3syg38X8MSyJ6M5PbXm9mIx+rhdP4LfqyH4oDWOtmtOrGFOBUgYd2zPZwN7shi1zF9rS0upfMMNCP3lkm4ErEYCmvrjd3XAtMbjkT2HVY8G8279RpQ8kt+ckKzBuZEmbXmiBSTlmr4RmA60qHpmKmJFdFTAgb0QkPDDHhgkduR3CpHclCW39C1tMtEl7hw19/4fVHTbhZN6AAA="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/kaggle/working/wave122")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave122-elementwise-profile-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave122.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave122.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("ptxas resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1",
                "GLCUDA_TELEMETRY": "1"}

    phase = "elementwise-profile"
    measured = run([exe, MODEL, "profile"], cwd=TREE, env=prod_env, check=False)
    save("elementwise-profile.log", measured)
    if measured.returncode or "[wave120-profile]" not in measured.stdout:
        raise RuntimeError("single-pass production profile failed")

    profile = json.loads(re.search(r"\[wave120-profile\]\s*(\{[^\n]+\})", measured.stdout).group(1))
    stages = [json.loads(x) for x in re.findall(r"\[wave120-stage\]\s*(\{[^\n]+\})", measured.stdout)]
    expected_names = [
        "qkv",
        "attn_norm",
        "attn_kv_write",
        "attention",
        "attn_out_quant",
        "ffn_down",
        "ffn_gate_up",
        "attn_out",
        "lm_head",
        "ffn_residual_norm_quant",
        "ffn_silu_quant",
        "ffn_residual_add",
    ]
    stage_names = [s["name"] for s in stages]
    if stage_names != expected_names or profile["gpu_prefill_ms"] <= 0:
        raise RuntimeError(f"event profile contract failed: {profile}, {stage_names}")
    if profile["oracle_token"] != 3323:
        raise RuntimeError(f"oracle token drifted: {profile}")

    stage_sum = sum(x["total_ms"] for x in stages)
    by_name = {x["name"]: x for x in stages}
    glue_names = ["attn_out_quant", "ffn_residual_norm_quant", "ffn_silu_quant", "ffn_residual_add"]
    ffn_names = ["ffn_down", "ffn_gate_up", "ffn_residual_norm_quant", "ffn_silu_quant",
                 "ffn_residual_add"]
    attention_names = ["qkv", "attn_norm", "attn_kv_write", "attention", "attn_out_quant",
                       "attn_out"]
    ranked = sorted(
        [
            {**s,
             "share_of_gpu_total": s["total_ms"] / profile["gpu_prefill_ms"],
             "share_of_stage_sum": s["total_ms"] / stage_sum if stage_sum else 0.0}
            for s in stages
        ],
        key=lambda s: -s["total_ms"],
    )
    glue_ranked = [s for s in ranked if s["name"] in glue_names]
    summary = {
        "wave": 122,
        "gpu": fields,
        "model": model_meta,
        "profile": profile,
        "stages": stages,
        "ranked_stages": ranked,
        "stage_sum_ms": stage_sum,
        "stage_sum_over_gpu_total": stage_sum / profile["gpu_prefill_ms"],
        "attention_ms": sum(by_name[n]["total_ms"] for n in attention_names),
        "ffn_ms": sum(by_name[n]["total_ms"] for n in ffn_names),
        "glue_ms": sum(by_name[n]["total_ms"] for n in glue_names),
        "glue_ranked": glue_ranked,
        "next_fusion_candidate": glue_ranked[0]["name"] if glue_ranked else None,
        "retention_authority": False,
        "target_15000_tps_achieved": profile["gpu_prefill_tps"] >= 15000,
    }
    report = [
        "# Wave 122 elementwise profile",
        "",
        f"- GPU prefill: {profile['gpu_prefill_ms']:.6f} ms over {profile['prompt_tokens']} tokens "
        f"= {profile['gpu_prefill_tps']:.1f} tok/s",
        f"- Stage sum: {stage_sum:.6f} ms ({summary['stage_sum_over_gpu_total']:.3f}x GPU total)",
        f"- Attention bucket: {summary['attention_ms']:.6f} ms",
        f"- FFN bucket: {summary['ffn_ms']:.6f} ms",
        f"- Glue subtotal: {summary['glue_ms']:.6f} ms",
        f"- Next fusion candidate: {summary['next_fusion_candidate']}",
        "",
        "| stage | ms | share of GPU total | calls | bytes read | macs |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for s in ranked:
        report.append(
            f"| `{s['name']}` | {s['total_ms']:.6f} | {100.0 * s['share_of_gpu_total']:.2f}% | "
            f"{s['calls']} | {s['bytes_read']} | {s['macs']} |"
        )
    (RESULTS / "wave122-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (RESULTS / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
    print("WAVE122_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
